# Template notebook

It's good to start with an introduction, to set the scene and introduce your audience to the data, and the problem you're solving as a team.

<br>

## Libraries
As always, we'll start by importing the necessary libraries.

In [1]:
# It's good practice to add comments to explain your code 
import numpy as np
import pandas as pd

**Question / Task 1**

Insert context about question / task 1 here.

In [2]:
# Add your code here
df = pd.read_csv(r"data/corona_tested_individuals_ver_006.english.csv")
print(df.head())

In [3]:
df.info()

In [4]:
# Display the first and last five rows
display(df.head())
display(df.tail())

# Check the dataset dimensions
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [5]:
# Column names
print(df.columns.tolist())

# Data types, non-null counts, and memory usage
df.info()


In [6]:
data_quality = pd.DataFrame({
    "Data type": df.dtypes,
    "Missing values": df.isna().sum(),
    "Missing percentage": (df.isna().mean() * 100).round(2),
    "Unique values": df.nunique()
}).sort_values("Missing percentage", ascending=False)

display(data_quality)

In [7]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)
print(
    "Duplicate percentage:",
    round(df.duplicated().mean() * 100, 2),
    "%"
)
#does not really tell me a lot, but why are there so many records when there should be 51k in March and 47k in April according to the original article?

In [8]:
display(df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
).head(20))

In [9]:
# Statistics for both numeric and categorical columns
display(df.describe(include="all").T)

In [10]:
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns

for column in categorical_columns:
    print(f"\nColumn: {column}")
    print(df[column].value_counts(dropna=False).head(10))

In [11]:
import matplotlib.pyplot as plt

numeric_columns = df.select_dtypes(include="number").columns

if len(numeric_columns) > 0:
    df[numeric_columns].hist(
        bins=20,
        figsize=(15, 10)
    )
    
    plt.tight_layout()
    plt.show()
else:
    print("No numeric columns were found.")

In [12]:
if len(numeric_columns) > 0:
    df[numeric_columns].plot(
        kind="box",
        subplots=True,
        layout=(-1, 3),
        figsize=(15, 4 * ((len(numeric_columns) + 2) // 3)),
        sharex=False,
        sharey=False
    )
    
    plt.tight_layout()
    plt.show()

In [13]:
if len(numeric_columns) >= 2:
    correlation_matrix = df[numeric_columns].corr()
    display(correlation_matrix.round(2))
else:
    print("At least two numeric columns are required.")

In [14]:
import seaborn as sns
import matplotlib.pyplot as plt

if len(numeric_columns) >= 2:
    plt.figure(figsize=(10, 8))
    
    sns.heatmap(
        df[numeric_columns].corr(),
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0
    )
    
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()

In [15]:
outlier_summary = {}

for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    outlier_summary[column] = (
        (df[column] < lower_limit) |
        (df[column] > upper_limit)
    ).sum()

outlier_summary = pd.Series(
    outlier_summary,
    name="Potential outliers"
).sort_values(ascending=False)

display(outlier_summary)

In [16]:
# Count missing values in each column
print(df.isna().sum())

# Total missing values
print(df.isna().sum().sum())

'''
Remove the unknown values for smaller portion of the categories
replace the unknown data as categorical data for age 60 or above
drop 'other' in corona result
Decide what models and run them and send them to Jerry
Discuss results and combine slides together
include graphs and visualisation
random seed =42
train-test = 70% 30%
train-validation-test = 70,15,15
'''

In [22]:
# Count and percentage of Unknown values
unknown_count = (df["fever"] == "Unknown").sum()
unknown_pct = unknown_count / len(df) * 100

print(f"Unknown values: {unknown_count}")
print(f"Percentage: {unknown_pct:.2f}%")

column = "Category"

unknown_pct = (df[column] == "Unknown").mean() * 100

if unknown_pct < 5:
    df = df[df[column] != "Unknown"].copy()
    print(f"Removed Unknown values ({unknown_pct:.2f}% of data)")
else:
    print(f"Unknown values make up {unknown_pct:.2f}% of data - consider imputation instead.")

In [ ]:
'''
The category "Unknown" accounted for a small proportion of observations and was removed to improve
data quality and reduce noise in subsequent analysis. The removal had minimal impact on the overall
dataset size and class distribution.

'''